# Python in the Smoke 💨 🐍 💨
## Applied notebook: web scraping, ethics, and London memorials

---

**Repo:** [`python-in-the-smoke`](https://github.com/MarkGotham/python-in-the-smoke)

**By:** Mark Gotham

**Licence:** MIT

**Colour and symbol key:** [Itemised and explained here](https://markgotham.github.io/python-in-the-smoke/#note-to-students)

**Assumed knowledge:**

- This notebook builds on the foundational Part I notebooks, using:
    - dictionaries, lists, loops, functions, `if` statements and f-strings.
    - pandas dataframes and category counts.
    - file paths to read and save files.
- No other applied notebook is required.

**Learning Objectives**: After completing this notebook, you will be able to ...
- Decide whether scraping is appropriate, and when to use a download or an API instead
- Check terms, `robots.txt` and personal-data issues before collecting anything
- Explain the structure of an HTML page
- Extract text and links from HTML with Beautiful Soup
- Write code that survives missing elements
- Collect a *small* set of pages politely: identified, slow and cached
- Say what a scraped set can and cannot show

---
## 🟢 Information: Why this topic, and why this design?

**The cultural question.**
Whose lives and what places does London choose to commemorate?
Plaques, statues and busts are public history.
We will ask how people connected to theatre are commemorated.
This continues the London theatre theme from the V&A API notebook, but you do not need that notebook.

**Scraping is the last resort, not the first move.**
Scraping means reading information out of web pages that were made for people, not programs.
It is fragile, and it puts load on someone else's server.
So we prefer, in this order: *ask → download → API → scrape*.
We scrape here because our example source offers no API that we know of, and its terms invite non-commercial re-use with acknowledgement.

**Why we practise on an invented page.**
Learning to parse HTML does not need a real website.
The pages in `sample_data/` are **invented** for this course: the people, plaques and places do not exist.
They have no copyright or privacy issues, work offline, and never change under you.
Only the final, optional section touches a real site, in tiny quantities and only once the checklist is satisfied.

📝 **For instructors:** before running the live section in class,
email the site owners (a template is in the optional exercises),
describe the scale, and ideally keep a written reply.
If you cannot or do not want to do this, everything before that section still works.

---
## *** 🟢 Information: Should I scrape? The decision ladder

Work down the ladder and stop at the first step that works.

| Step | Question | If yes |
|---|---|---|
| 1. Ask | Would the owner simply share the data? | Ask. It is often quickest and gives better data. |
| 2. Download | Is there a CSV or data download (e.g. open data portals)? | Use it. |
| 3. API | Is there an API (as in the V&A notebook)? | Use it. |
| 4. Scrape | None of the above, *and* the terms allow it? | Scrape small, slowly and politely. |
| ✋ Stop | Do the terms forbid it, is it behind a login or paywall, or does it involve private personal data? | Do not scrape. |

---
### 🟣 Core Exercise: Apply the ladder

For each scenario, write one line in the Markdown cell below: which step, and why?

1. Object records from the V&A collection, which has a public search API.
2. A borough council publishes its list of public libraries as a CSV on its open data page.
3. A volunteer-run website catalogues London memorials. It has no API and says non-commercial re-use is welcome with acknowledgement.
4. A ticketing site shows seat prices. Its terms forbid automated collection.

*Your four answers here.*

---
---
---

### ✅ Solution: Apply the ladder

1. **API** (step 3). The V&A provides one, so use it.
2. **Download** (step 2). Download the CSV; scraping the page would be pointless extra work.
3. **Scrape, small and politely** (step 4), after checking the terms and ideally asking. This is our case.
4. **Stop** (✋). The terms forbid it. Do not scrape; look for an official data source or ask.

Notice that "technically possible" never appears in the ladder.

---
## *** 🟢 Information: Ethics checklist

Run through this *every time*, before writing any scraping code.

| Check | Where to look | If "no" |
|---|---|---|
| **Terms and copyright:** does the site allow this use? | Its copyright policy or terms of use | Stop, or ask permission |
| **`robots.txt`:** are these pages allowed for automated access? | `https://site/robots.txt` | Stop |
| **Personal data:** do pages describe living, identifiable people? | Read a sample of pages | Collect only what you need, or stop (UK GDPR applies to living people) |
| **Load:** will your requests be a burden? | Your plan | Fewer pages, longer delays, cache everything |
| **Identity:** can the owner tell who you are? | Your `User-Agent` | Add a contact email |
| **Credit and sharing:** do you know what you may publish? | The terms | Attribute, and do not republish pages |

📝 Notes:
- `robots.txt` is a convention, not a permission. Terms of use and copyright still apply even where `robots.txt` allows access.
- Never bypass logins, blocks or rate limits. If a site asks you to stop, stop.
- Images on a page are often owned by someone else. Even when a site allows re-use of its own text, it may not authorise re-use of images it borrowed. We collect text only.
- Obviously this is simply a checklist for this course, not legal advice.

---
## 🟢 Information: Our source

Our real-world example is [London Remembers](https://www.londonremembers.com/),
a volunteer-run catalogue of London's plaques, statues and memorials.
Its [copyright policy](https://www.londonremembers.com/about/copyright) describes it as non-commercial and educational.
At the time of writing, it says it is generally happy for you to use excerpts of its own text in non-commercial publications,
provided you credit "LondonRemembers.com" and let them know.
It also does **not** authorise re-use by others
of material that it has sourced from other websites.

To check again before you start:
- Still no API or bulk download?
- Any changes to the policies?

---
## 🟢 Information: Set up

We need two packages:
- `requests` fetches pages (as in the API notebook).
- `beautifulsoup4` reads HTML.
Note that the install name and the import name differ: we install `beautifulsoup4` but import from `bs4`.

Run the cells below.
To review installation and import, see Notebook f.

In [ ]:
%pip install requests beautifulsoup4

In [ ]:
from pathlib import Path
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

📝 **On Colab:** this notebook reads the invented pages in `sample_data/` (`mock_robots.txt` and the `mock_*.html` files).
Download them from the repo and upload them to the session first, into the `sample_data` folder
(Colab already creates one, so add them alongside whatever is there).
Files uploaded to a Colab session are deleted when it ends,
so you will need to upload them again next time.
Running the cell below checks that they arrived.

In [ ]:
print(sorted(p.name for p in Path("sample_data").glob("mock_*")))

---
## 🟢 Information: Reading `robots.txt`

A site's `robots.txt` file lists which paths automated programs should avoid, and sometimes how long to wait between requests.
Python can read one with `RobotFileParser` and answer the question "may I fetch this path?"

Here is a short **invented** example from `sample_data/mock_robots.txt`:

In [ ]:
mock_robots = Path("sample_data/mock_robots.txt").read_text(encoding="utf-8")
print(mock_robots)

In [ ]:
rp_mock = RobotFileParser()
rp_mock.parse(mock_robots.splitlines())

print(rp_mock.can_fetch("my-course-bot", "/memorials/example"))
print(rp_mock.crawl_delay("my-course-bot"))

`can_fetch(user_agent, path)` returns `True` or `False`.
`crawl_delay` is the number of seconds the site asks us to wait (or `None` if it does not say).

---
### 🟣 Core Exercise: Check some paths

Using `rp_mock`, check whether `my-course-bot` may fetch:
- `/private/notes`
- `/subjects/example`

---
---
---

### ✅ Solution: Check some paths

In [ ]:
print(rp_mock.can_fetch("my-course-bot", "/private/notes"))
print(rp_mock.can_fetch("my-course-bot", "/subjects/example"))

The first is `False` (disallowed) and the second is `True`.
We return to the real thing in the live section.

---
## *** 🟢 Information: HTML is nested structure

A web page is text with **tags** that mark up its structure.
You have seen this idea before: JSON nests dictionaries and lists, and HTML nests elements inside elements.

```
<article class="memorial">
    <h1 class="memorial-title">Plaque to Ada Quill</h1>
    <p class="location">Fictional Lane, WC2</p>
</article>
```

- `<h1>` and `<p>` are **tags**. Most come as an opening tag and a closing tag (`</h1>`).
- `class="location"` is an **attribute**. Here it labels *what kind* of paragraph this is.
- The `<p>` and `<h1>` are *inside* the `<article>`, so they are its children.

Open one of the invented pages and read its raw HTML:

In [ ]:
html = Path("sample_data/mock_memorial_1.html").read_text(encoding="utf-8")
print(html)

📝 In your own browser, you can see the same thing for any real page: right-click and choose **View page source** or **Inspect**.
This is the main way to work out where the information you want sits.

---
### 🟣 Core Exercise: Read the HTML

Read the HTML printed above, then answer in a Markdown cell:
1. Which tag and class hold the location?
2. How is the list of subjects marked up?
3. Where does the inscription appear?

*Your answers here.*

---
---
---

### ✅ Solution: Read the HTML

1. A `<p>` tag with `class="location"`.
2. A `<ul class="subjects">` list. Each subject is an `<li class="subject">` containing a link (`<a href=...>`) and a `<span class="role">`.
3. In a `<blockquote class="inscription">`, with line breaks written as `<br>`.

You have just done what every scraper does by hand first: found the pattern in the markup.

---
## 🟢 Information: Parsing with Beautiful Soup

Beautiful Soup turns HTML text into an object we can search.
- `soup.find("p", class_="location")` finds the first `<p>` with that class. (`class_` has an underscore because `class` is a reserved word in Python.)
- `soup.select("ul.subjects li")` finds *all* matches, using CSS-style selectors, and returns a list.
- `.get_text(strip=True)` gives the text inside, with surrounding whitespace removed.
- `tag["href"]` reads an attribute.

In [ ]:
soup = BeautifulSoup(html, "html.parser")
print(soup.title.get_text())
print(soup.find("h1").get_text(strip=True))
print(soup.find("p", class_="location").get_text(strip=True))

In [ ]:
for li in soup.select("ul.subjects li"):
    print(li.a.get_text(strip=True), "|", li.a["href"])

---
### 🟣 Core Exercise: Extract one memorial

From `soup`, extract and print:
- the title (the `<h1>`)
- the location
- the memorial type (`<p class="type">`)
- a list of the subjects' names, built with a loop

---
---
---

### ✅ Solution: Extract one memorial

In [ ]:
title = soup.find("h1").get_text(strip=True)
location = soup.find("p", class_="location").get_text(strip=True)
memorial_type = soup.find("p", class_="type").get_text(strip=True)

subjects = []
for li in soup.select("ul.subjects li"):
    subjects.append(li.a.get_text(strip=True))

print(title)
print(location)
print(memorial_type)
print(subjects)

Notice that every line depends on the page's exact structure.
If the site changes its layout, selectors like these stop working.
That fragility is a core reason to prefer APIs.

---
## *** 🟢 Information: Pages are not all alike

A function lets us apply the same extraction to many pages.
But real pages are inconsistent: some fields are missing, and text has stray whitespace.

- `soup.find(...)` returns `None` when nothing matches, and calling `.get_text()` on `None` fails.
- So we check `if element is None` first.
- `" ".join(text.split())` collapses any run of spaces and line breaks into single spaces.

Page 3 of our invented set deliberately has **no inscription**, and page 2 has messy spacing.

---
### 🟣 Core Exercise: A function that survives

Write `read_memorial(path)` that opens a page and returns a dictionary with the keys `title`, `location`, `memorial_type`, `subjects` (a list) and `inscription`.
- If there is no inscription, set it to `None`.
- Clean the inscription's whitespace.

Test it on `sample_data/mock_memorial_2.html` and `sample_data/mock_memorial_3.html`.

---
---
---

### ✅ Solution: A function that survives

In [ ]:
def read_memorial(path):
    page = BeautifulSoup(Path(path).read_text(encoding="utf-8"), "html.parser")

    subjects = []
    for li in page.select("ul.subjects li"):
        subjects.append(li.a.get_text(strip=True))

    inscription_tag = page.find("blockquote", class_="inscription")
    if inscription_tag is None:
        inscription = None
    else:
        inscription = " ".join(inscription_tag.get_text(" ").split())

    return {
        "title": page.find("h1").get_text(strip=True),
        "location": page.find("p", class_="location").get_text(strip=True),
        "memorial_type": page.find("p", class_="type").get_text(strip=True),
        "subjects": subjects,
        "inscription": inscription,
    }

In [ ]:
print(read_memorial("sample_data/mock_memorial_2.html"))
print(read_memorial("sample_data/mock_memorial_3.html"))

A missing inscription becomes `None` rather than a crash, and we have recorded that it was missing instead of hiding it.
The inscription's `<br>` line breaks are handled by `get_text(" ")`, which puts a space where the tags were.
For a fuller version, you would check *every* field this way.

---
## 🟢 Information: From a listing to many pages

Real sites have **listing pages** that link to **detail pages**.
The usual pattern is:
1. Read the listing and collect the links.
2. Visit each detail page (with a delay, if it is a real server).
3. Build a table.

Here the whole thing runs on local invented files, so no delay is needed. On a real site you would add one (see the live section).

---
### 🟣 Core Exercise: Build a table

1. Parse `sample_data/mock_listing.html`.
2. Collect the `href` of every link inside `ul.memorial-list`.
3. Use a loop and `read_memorial` to read each page.
4. Build a dataframe and count the memorial types.

---
---
---

### ✅ Solution: Build a table

In [ ]:
listing = BeautifulSoup(Path("sample_data/mock_listing.html").read_text(encoding="utf-8"), "html.parser")

links = []
for a in listing.select("ul.memorial-list a"):
    links.append(a["href"])
print(links)

In [ ]:
rows = []
for link in links:
    rows.append(read_memorial(Path("sample_data") / link))

memorials = pd.DataFrame(rows)
memorials

In [ ]:
memorials["memorial_type"].value_counts()

The listing said "Showing 4 of 4", and we read 4 pages, so we know our table is complete *for this invented set*.
Real sites often report a total, which is worth comparing against the number of rows you collected.

---
### 🟣 Core Exercise: What can the table support?

Imagine the same table came from a real site, with 15 rows.
Write three short sentences in the Markdown cell below:
1. State one thing the table could show.
2. Suggest a cultural question it could help investigate.
3. State one limit of treating it as evidence.

*Your three sentences here.*

---
---
---

### ✅ Solution: What can the table support?

Use your own values.

Example structure:
- "In these [n] pages, [type] is the most common memorial type."
- "That could help us ask [a specific cultural question, e.g. who is commemorated with a plaque and who with a statue]."
- "This is a small, selected set, not a representative sample of London's memorials."

What a small scraped set **cannot** establish:
- How common any type is across London.
- Anything about people or places the catalogue left out. Read the site's own inclusion criteria.
- That separate pages are independent (one person can appear on several).

Your counts describe what you collected, and the collection depends on who made the catalogue, what they chose to include, and what you chose to scrape.

---
## *** 🟢 Information: The live section (optional, gated)

Everything above ran on invented pages.
This section touches a real site, in tiny quantities, and **only** if you are ready.

Before setting `LIVE = True`, tick every item:
- [ ] I have read the site's current copyright policy and terms.
- [ ] My instructor or I have asked the owners, and I know what we may do.
- [ ] I have put my contact email in `USER_AGENT`.
- [ ] I understand I will fetch at most a couple of pages, slowly, and cache them.
- [ ] I will collect text only (no images), and I will credit LondonRemembers.com and not republish pages.

📝 The cell below does nothing unless you change `LIVE`. The `YOUR-EMAIL-HERE` placeholder also stops it until you replace it.

In [ ]:
LIVE = False  # change to True only after the checklist above
USER_AGENT = "python-in-the-smoke-course (contact: YOUR-EMAIL-HERE)"

BASE = "https://www.londonremembers.com"
PAGE_URL = BASE + "/memorials/mercantile-marine-memorial-ww1"


def ready_for_live():
    if not LIVE:
        print("LIVE is False, so nothing was requested.")
        return False
    if "YOUR-EMAIL-HERE" in USER_AGENT:
        print("Put your contact email in USER_AGENT first.")
        return False
    return True

**Step 1: `robots.txt`.**
We fetch it (one small request) and ask whether our page is allowed.
We also use any crawl delay the site asks for, with a minimum of 5 seconds of our own.

In [ ]:
robots_ok = False
delay = 5

if ready_for_live():
    robots = requests.get(BASE + "/robots.txt", headers={"User-Agent": USER_AGENT}, timeout=30)
    print("robots.txt status:", robots.status_code)
    if robots.status_code == 200:
        rp = RobotFileParser()
        rp.parse(robots.text.splitlines())
        robots_ok = rp.can_fetch(USER_AGENT, PAGE_URL)
        delay = max(5, rp.crawl_delay(USER_AGENT) or 0)
    else:
        print("No usable robots.txt. Check the site manually, and ask before continuing.")
    print("Allowed by robots.txt:", robots_ok)

**Step 2: one page, politely, cached.**
`fetch_cached` waits, identifies us, checks for errors, and **saves the HTML** so we never request the same page twice.
This is the same "collect once, analyse many times" idea as saving the API response.

In [ ]:
def fetch_cached(url, cache_dir="html_cache", delay=5):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(exist_ok=True)
    path = cache_dir / (url.rstrip("/").split("/")[-1] + ".html")

    if path.exists():
        print("Using cached copy:", path)
        return path.read_text(encoding="utf-8")

    time.sleep(delay)
    response = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)
    response.raise_for_status()
    path.write_text(response.text, encoding="utf-8")
    print("Fetched and cached:", path)
    return response.text

In [ ]:
if robots_ok:
    live_html = fetch_cached(PAGE_URL, delay=delay)
    live_soup = BeautifulSoup(live_html, "html.parser")
    print(live_soup.title.get_text(strip=True))
    h1 = live_soup.find("h1")
    print(h1.get_text(strip=True) if h1 else "No <h1> found")

📝 We only use generic tags (`<title>`, `<h1>`) here, because every page has them.
The site's own markup will differ from our invented pages, so any other selector must come from *your* inspection of the real page.

---
### ⭐ Optional Exercise: Inspect a real page

1. Open `PAGE_URL` in your browser and use **Inspect** on one piece of information (e.g. the location).
2. Write the selector you found in a Markdown cell.
3. Test it on `live_soup` with `live_soup.select_one(...)`.
4. Say what would happen to your code if the site were redesigned.

---
---
---

### ✅ Solution: Inspect a real page

There is no fixed solution: selectors are specific to the site and can change at any time.
The pattern is:

In [ ]:
# Replace the selector with the one you found by inspecting the page.
if robots_ok:
    element = live_soup.select_one("YOUR-SELECTOR-HERE")
    print(element.get_text(strip=True) if element else "Selector matched nothing")

If the site were redesigned, `select_one` would return `None`, the output would say "matched nothing", and you would have to inspect the page again.
That is normal for scraping, and the reason to check for `None` every time.

---
### ⭐ Optional Exercise: A second page, still politely

Fetch one more page, `BASE + "/memorials/mercantile-marine-memorial-ww2"`, in the same way.
Check `robots.txt` permission first, use the delay and cache, and print each page's `<h1>`.
Do not fetch more pages than this without asking the site owners.

---
---
---

### ✅ Solution: A second page, still politely

In [ ]:
urls = [PAGE_URL, BASE + "/memorials/mercantile-marine-memorial-ww2"]

if robots_ok:
    for url in urls:
        if not rp.can_fetch(USER_AGENT, url):
            print("Not allowed by robots.txt:", url)
            continue
        page_html = fetch_cached(url, delay=delay)
        page_soup = BeautifulSoup(page_html, "html.parser")
        print(page_soup.find("h1").get_text(strip=True))

Notice the order: check permission, wait, fetch, cache, parse.
Cached pages cost the site nothing on later runs.
If you want to keep working, use the cache and change your analysis, not your requests.

---
### ⭐ Optional Exercise: Ask permission

The site's policy asks users to let the owners know about their use.
Write a short email that:
- says who you are and what the course or project is
- says exactly what you will collect, how much, and how slowly
- says you will credit the source and not republish pages
- asks them to say if they would prefer you not to

*Your draft here.*

---
---
---

### ✅ Solution: Ask permission

Adapt this template:

> **Subject:** Teaching use of London Remembers pages (python-in-the-smoke)
>
> Dear London Remembers team,
>
> I teach/use an open introductory Python course on London cultural data ([python-in-the-smoke](https://github.com/MarkGotham/python-in-the-smoke)). In one notebook, students learn web scraping and its ethics, using your site as the real-world example. Your copyright policy says non-commercial use with acknowledgement is welcome and asks users to let you know, so I am writing to do that.
>
> We would fetch only [N] pages, each once, with a [5]-second delay and a User-Agent that includes my contact email, caching them locally. We would collect text only (no images), credit LondonRemembers.com, and not republish your pages.
>
> Please tell me if you would prefer that we did not, or if there is a better way for us to work with your data.
>
> With thanks,
> [Name, role, email]

A reply (or the lack of an objection) does not remove your own responsibility to follow the policy.
If they say no, respect it.

---
## ⚠️ Common Errors

| Symptom | What to check |
|---|---|
| `ModuleNotFoundError: No module named 'bs4'` | Install `beautifulsoup4` (the import name is `bs4`) |
| `AttributeError: 'NoneType' object has no attribute ...` | `find` matched nothing: check the selector and test `if element is None` |
| `IndexError` after `select(...)[0]` | The list is empty: the selector is wrong, or the content is loaded by JavaScript |
| `HTTPError` 403 or 429 | Forbidden, or too many requests: stop, slow down, check terms and `robots.txt` |
| `FileNotFoundError` for `sample_data/...` | Check the working directory with `Path.cwd()` (Notebook g) |
| `UnicodeDecodeError` | Pass `encoding="utf-8"` when reading and writing files |
| Old content after the site changed | Your cached copy is out of date: delete that cached file deliberately and re-fetch only if allowed |

**The cells below fail on purpose.**
Read each message, work out the cause, then compare with the explanation underneath.

### ⚠️ Deliberate error 1: a missing element

In [ ]:
page3 = BeautifulSoup(Path("sample_data/mock_memorial_3.html").read_text(encoding="utf-8"), "html.parser")
page3.find("blockquote", class_="inscription").get_text()

`AttributeError: 'NoneType' object has no attribute 'get_text'`: page 3 has no inscription, so `find` returned `None`.
Check `if tag is None` before using the result, as `read_memorial` does.

### ⚠️ Deliberate error 2: the server says stop

In [ ]:
fake = requests.models.Response()
fake.status_code = 429
fake.reason = "Too Many Requests"
fake.url = "https://example.org/page"
fake.raise_for_status()

`HTTPError: 429 Client Error`: this simulates a server telling us we are asking too often.
The answer is not to retry harder. Slow down, rely on the cache, and ask the owners if you need more.

### ⚠️ Deliberate error 3: content that is not in the HTML

In [ ]:
js_page = BeautifulSoup('<html><body><div id="app"></div><script src="app.js"></script></body></html>', "html.parser")
js_page.select("li.subject")[0]

`IndexError: list index out of range`: some pages build their content with JavaScript *after* loading, so the HTML we fetch is nearly empty.
Beautiful Soup cannot see that content. This is another sign to look for an API or data download instead.

---
## 🟢 Information: Summary

| To ... | Write ... |
|---|---|
| Read `robots.txt` rules | `rp = RobotFileParser(); rp.parse(text.splitlines())` |
| Check a path is allowed | `rp.can_fetch(user_agent, url)` |
| Identify yourself | `requests.get(url, headers={"User-Agent": ...}, timeout=30)` |
| Parse HTML | `soup = BeautifulSoup(html, "html.parser")` |
| Find one / many elements | `soup.find("p", class_="x")` / `soup.select("ul li")` |
| Get text / an attribute | `tag.get_text(strip=True)` / `tag["href"]` |
| Survive missing elements | `if tag is None:` |
| Be polite | `time.sleep(delay)`, a small cap, and a cache |

Scraping is a last resort: ask, download, or use an API first.
Collect the least you need, as slowly as is reasonable, credit the source, and keep your claims within what a small, selected set can show.

**Final reflection:**
- Which step of the decision ladder would you try first for your own project?
- What would you want to know about how a catalogue was compiled before drawing conclusions from it?
- What would make you decide *not* to scrape?

### Source and documentation

- [London Remembers copyright policy](https://www.londonremembers.com/about/copyright)
- [Beautiful Soup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Requests quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [Python `urllib.robotparser`](https://docs.python.org/3/library/urllib.robotparser.html)
- [Robots Exclusion Protocol (RFC 9309)](https://www.rfc-editor.org/rfc/rfc9309)

The pages in `sample_data/` (`mock_*.html`, `mock_robots.txt`) are invented for this course. The people, memorials and places in them do not exist.
Real material from London Remembers is not included in this repository. Credit "LondonRemembers.com" if you use any of its text, and tell them about your use.

This notebook is educational, not legal advice.

End

---